In [1]:
import torch

print("PyTorch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA :", torch.version.cuda)
else:
    print("GPU non disponible")

PyTorch version : 2.11.0+cu128
CUDA disponible : True
GPU : Tesla T4
CUDA : 12.8


In [2]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 17.4 MB/s eta 0:00:00


In [3]:
import torch

print("CUDA :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))

CUDA : True
GPU : Tesla T4


In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cpu"
)

print("BGE-M3 chargé avec succès !")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

BGE-M3 chargé avec succès !


In [5]:
texts = [
    "Les fissures longitudinales sont orientées parallèlement à l'axe de la chaussée.",
    "Les nids-de-poule sont des dégradations localisées de la chaussée.",
    "La maintenance routière comprend différentes méthodes de réparation."
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

print("Shape :", embeddings.shape)

Shape : (3, 1024)


In [6]:
query = "Qu'est-ce qu'une fissure longitudinale ?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

similarities = embeddings @ query_embedding.T

print(similarities)

[[0.63592064]
 [0.41683856]
 [0.34842753]]


In [7]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_name = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

print("Qwen chargé avec succès !")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen chargé avec succès !


In [8]:
print(
    f"GPU utilisé : {torch.cuda.get_device_name(0)}"
)

print(
    f"VRAM utilisée : "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

GPU utilisé : Tesla T4
VRAM utilisée : 5.18 GB


In [9]:
messages = [
    {
        "role": "system",
        "content": "Tu es un assistant spécialisé dans les dommages routiers."
    },
    {
        "role": "user",
        "content": "Explique brièvement ce qu'est une fissure longitudinale dans une route."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    [text],
    return_tensors="pt"
).to(llm.device)

with torch.no_grad():
    outputs = llm.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Une fissure longitudinale dans une route est un type de dommage qui se forme le long de la direction principale de la circulation, généralement parallèle aux bords de la chaussée. Ce type de fissure est souvent caractérisé par des lignes rectilignes qui s'étendent de l'extérieur vers le centre de la voie. Ces fissures sont généralement le résultat de la pression continue exercée sur la route par le trafic automobile et peuvent être aggravées par des conditions météorologiques défavorables comme la sécheresse ou les gelés successifs. Elles peuvent aussi indiquer que la couche de revêtement de la route est en train de s'épaissir ou de s'affaisser, ce qui peut entraîner une perte de stabilité de la route et potentiellement des risques pour les conducteurs.


In [10]:
documents = [
    """
    Les fissures longitudinales sont des fissures orientées
    parallèlement à l'axe de la chaussée. Elles peuvent être
    associées à des problèmes de structure ou de vieillissement.
    """,

    """
    Les nids-de-poule sont des cavités localisées dans la chaussée.
    Leur formation peut être favorisée par l'eau, le trafic et
    la dégradation progressive du revêtement.
    """,

    """
    La maintenance routière comprend l'inspection, la détection
    des dégradations, la réparation et le suivi de l'état
    des infrastructures.
    """
]

In [11]:
doc_embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True
)

print(doc_embeddings.shape)

(3, 1024)


In [12]:
question = "Quels sont les problèmes associés aux fissures longitudinales ?"

question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

In [13]:
scores = doc_embeddings @ question_embedding.T

best_index = scores[:, 0].argmax()

print("Document sélectionné :")
print(documents[best_index])

print("Score :", scores[best_index, 0])

Document sélectionné :

    Les fissures longitudinales sont des fissures orientées
    parallèlement à l'axe de la chaussée. Elles peuvent être
    associées à des problèmes de structure ou de vieillissement.
    
Score : 0.6910379


Etape 6 : FAISS

In [14]:
import faiss
import numpy as np

# FAISS utilise des float32
doc_embeddings = np.asarray(
    doc_embeddings,
    dtype="float32"
)

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(doc_embeddings)

print("Nombre de vecteurs :", index.ntotal)
print("Dimension :", dimension)

Nombre de vecteurs : 3
Dimension : 1024


In [15]:
question = "Quels sont les problèmes associés aux fissures longitudinales ?"

question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

question_embedding = np.asarray(
    question_embedding,
    dtype="float32"
)

scores, indices = index.search(
    question_embedding,
    k=2
)

print("Indices :", indices)
print("Scores :", scores)

Indices : [[0 1]]
Scores : [[0.6910379  0.37944818]]


In [16]:
for rank, idx in enumerate(indices[0]):
    print(f"\n--- Résultat {rank + 1} ---")
    print(documents[idx])
    print("Score :", scores[0][rank])


--- Résultat 1 ---

    Les fissures longitudinales sont des fissures orientées
    parallèlement à l'axe de la chaussée. Elles peuvent être
    associées à des problèmes de structure ou de vieillissement.
    
Score : 0.6910379

--- Résultat 2 ---

    Les nids-de-poule sont des cavités localisées dans la chaussée.
    Leur formation peut être favorisée par l'eau, le trafic et
    la dégradation progressive du revêtement.
    
Score : 0.37944818


In [17]:
import os

BASE_DIR = "/content/road_damage_rag"
DATA_DIR = os.path.join(BASE_DIR, "data")
VECTOR_DIR = os.path.join(BASE_DIR, "vector_db")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VECTOR_DIR, exist_ok=True)

print("Dossiers créés.")

Dossiers créés.


In [18]:
import os

for filename in os.listdir(DATA_DIR):
    print(filename)

In [19]:
import os

DATA_DIR = "/content/road_damage_rag/data"

documents = []

for filename in os.listdir(DATA_DIR):

    if filename.lower().endswith(".txt"):

        path = os.path.join(DATA_DIR, filename)

        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        documents.append({
            "text": text,
            "source": filename
        })

print("Documents chargés :", len(documents))

for doc in documents:
    print("-", doc["source"])

Documents chargés : 6
- Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt
- Pathologie_des_routes_contenu_organise.txt
- Cours_Pathologies_Entretien_Routes_Ouvrages_contenu_organise.txt
- Distress_Identification_Manual_contenu_organise.txt
- RDD2022_contenu_organise.txt
- ArticleOuargla_contenu_organise.txt


Découper les documents en chunks

Maintenant on découpe chaque document en morceaux.

Je te conseille de commencer avec :

chunk_size = 800
overlap = 100

In [20]:
def chunk_text(text, chunk_size=800, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [21]:
chunks = []

for doc in documents:

    doc_chunks = chunk_text(
        doc["text"],
        chunk_size=800,
        overlap=100
    )

    for chunk in doc_chunks:

        chunks.append({
            "text": chunk,
            "source": doc["source"]
        })

print("Nombre total de chunks :", len(chunks))

Nombre total de chunks : 479


In [22]:
for i in range(min(3, len(chunks))):

    print("=" * 60)
    print("CHUNK", i + 1)
    print("SOURCE :", chunks[i]["source"])
    print(chunks[i]["text"][:500])

CHUNK 1
SOURCE : Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt
===== PAGE 1 =====



===== PAGE 2 =====



===== PAGE 3 =====

PROJET

===== PAGE 4 =====

PROJET

Catalogage avant publication de la Bibliothèque nationale du Canada Catalogage avant publication de la Bibliothèque nationale du Canada Catalogage avant publication de la Bibliothèque nationale du Canada Catalogage avant publication de la Bibliothèque nationale du Canada Catalogage avant publication de la Bibliothèque nationale du Canada

Vedette principale au titre :

Manuel d’identification des 
CHUNK 2
SOURCE : Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt
N 2-551-21654-0

1. Chaussées souples – Évaluation. 2. Chaussées – Dégradations – Évaluation. 3. Chaussées – Fissuration – Évaluation. 4. Chaussées souples – Entretien et réparations. I. Québec (Province). Ministère des transports.

TE270.M36 2002 625.8’5’0288 C2002-94l901-8

===== PAGE 5 =====

PROJET

===== PA

Embeddings avec BGE-M3

Maintenant on transforme les chunks, et non les documents entiers, en vecteurs.

In [25]:
texts = [
    chunk["text"]
    for chunk in chunks
]

chunk_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

In [26]:
import numpy as np

chunk_embeddings = np.asarray(
    chunk_embeddings,
    dtype="float32"
)

print("Shape :", chunk_embeddings.shape)

Shape : (479, 1024)


Construire FAISS

Maintenant on donne ces vecteurs à FAISS :

In [27]:
import faiss

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(chunk_embeddings)

print("FAISS prêt.")
print("Nombre de vecteurs :", index.ntotal)
print("Dimension :", dimension)

FAISS prêt.
Nombre de vecteurs : 479
Dimension : 1024


In [28]:
def retrieve(query, k=5):

    # Transformer la question en vecteur
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # Recherche FAISS
    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score)
        })

    return results

In [29]:
question = "Qu'est-ce qu'une fissure longitudinale ?"

results = retrieve(question, k=5)

for i, result in enumerate(results):

    print("=" * 70)
    print(f"RESULTAT {i+1}")
    print("Source :", result["source"])
    print("Score :", round(result["score"], 4))
    print()
    print(result["text"][:800])

RESULTAT 1
Source : Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt
Score : 0.6212

gradations

○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○

2.1 F i s s u r a t i o n

PROJET

Fissur Fissur Fissur Fissur Fissures longitudinales es longitudinales es longitudinales es longitudinales es longitudinales (hors piste de roues)

Description

Rupture du revêtement relativement parallèle à la direction de la route, excluant les fissures de gel, en dehors des pistes de roues, soit dans les bandes numéros 1, 3 et 5 (voir figure 1–1).

Causes probables

· Joint de construction mal exécuté le long de la travée adjacente.

· Ségrégation de l’enrobé à la pose (ex. : centre de l’épandeur).

· Vieillissement du revêtement.

Niveau de sévérité

Faible aible aible aible aible : Fissures simples et intermittentes dont les ouvertures sont inférieures à 5 mm. Les bords sont en général francs et bien définis.
RESULTAT 2
Source : Cours_Pathologies_Entretien_Routes_Ouvrages_contenu_organise.txt
Sc

In [30]:
def build_context(results):

    context = ""

    for i, result in enumerate(results):

        context += f"""
SOURCE {i+1}: {result['source']}

{result['text']}

------------------------------
"""

    return context

In [31]:
context = build_context(results)

print(context)


SOURCE 1: Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt

gradations

○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○ ○

2.1 F i s s u r a t i o n

PROJET

Fissur Fissur Fissur Fissur Fissures longitudinales es longitudinales es longitudinales es longitudinales es longitudinales (hors piste de roues)

Description

Rupture du revêtement relativement parallèle à la direction de la route, excluant les fissures de gel, en dehors des pistes de roues, soit dans les bandes numéros 1, 3 et 5 (voir figure 1–1).

Causes probables

· Joint de construction mal exécuté le long de la travée adjacente.

· Ségrégation de l’enrobé à la pose (ex. : centre de l’épandeur).

· Vieillissement du revêtement.

Niveau de sévérité

Faible aible aible aible aible : Fissures simples et intermittentes dont les ouvertures sont inférieures à 5 mm. Les bords sont en général francs et bien définis.

------------------------------

SOURCE 2: Cours_Pathologies_Entretien_Routes_Ouvrages_contenu_organise.txt

oh

In [54]:
def build_prompt(question, context):

    return f"""
Tu es un assistant spécialisé dans les dommages et pathologies routières.

Réponds à la question uniquement à partir du contexte fourni.

Règles :
- Utilise uniquement les informations présentes dans le contexte.
- N'invente aucune information.
- Si l'information n'est pas présente dans le contexte,
  indique clairement que l'information n'a pas été trouvée.
- Donne une réponse claire, précise et concise.
- Ne mentionne pas les sources dans ta réponse.
- Ne mentionne pas "SOURCE 1", "SOURCE 2", etc.
- Les sources seront affichées séparément par l'application.

Contexte :
{context}

Question :
{question}

Réponse :
"""

In [55]:
def generate_answer(question, context):

    prompt = build_prompt(
        question,
        context
    )

    messages = [
        {
            "role": "system",
            "content": "Tu es un assistant spécialisé dans les dommages routiers."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(llm.device)

    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer

In [56]:
def ask_rag(question, k=5):

    # 1. Recherche des chunks
    results = retrieve(
        question,
        k=k
    )

    # 2. Construction du contexte
    context = build_context(
        results
    )

    # 3. Génération avec Qwen
    answer = generate_answer(
        question,
        context
    )

    return answer, results

Premier vrai test

In [57]:
question = "Qu'est-ce qu'une fissure longitudinale et quelles sont ses caractéristiques ?"

answer, sources = ask_rag(
    question,
    k=5
)

print("========== RÉPONSE ==========")
print(answer)

print("\n========== SOURCES ==========")

for source in sources:

    print(
        source["source"],
        "→ score :",
        round(source["score"], 4)
    )

========== RÉPONSE ==========
Une fissure longitudinale est une rupture du revêtement relativement parallèle à la direction de la route, en dehors des pistes de roues, soit dans les bandes numéros 1, 3 et 5. Elle est principalement due à la fatigue de la structure de la chaussée. Les fissures longitudinales simples et intermittentes ont des ouvertures inférieures à 5 mm et leurs bords sont généralement francs et bien définis. Elles peuvent évoluer vers des déformations et des arrachements, formant un réseau de fissures appelé faïençage. Les causes probables incluent un joint de construction mal exécuté, une ségrégation de l'enrobé à la pose, et le vieillissement du revêtement.

========== SOURCES ==========
Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt → score : 0.6195
Cours_Pathologies_Entretien_Routes_Ouvrages_contenu_organise.txt → score : 0.5831
Cours_Pathologies_Entretien_Routes_Ouvrages_contenu_organise.txt → score : 0.5826
Cours_Pathologies_Entretie

In [36]:
print("GPU :", torch.cuda.get_device_name(0))
print("LLM device :", llm.device)

GPU : Tesla T4
LLM device : cuda:0


In [37]:
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

GPU memory allocated: 5.19 GB
GPU memory reserved: 6.05 GB


In [38]:
print("GPU :", torch.cuda.get_device_name(0))
print("LLM device :", llm.device)
print("First parameter :", next(llm.parameters()).device)

GPU : Tesla T4
LLM device : cuda:0
First parameter : cuda:0


In [ ]:
# import gc
# import torch

# del llm
# gc.collect()
# torch.cuda.empty_cache()

# print("VRAM après nettoyage :", torch.cuda.memory_allocated() / 1024**3, "GB")

In [39]:
from transformers import AutoModelForCausalLM

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [40]:
print("GPU :", torch.cuda.get_device_name(0))
print("LLM device :", llm.device)
print("First parameter :", next(llm.parameters()).device)

GPU : Tesla T4
LLM device : cuda:0
First parameter : cuda:0


# l’étape des tests RAG.

In [41]:
test_questions = [
    "Qu'est-ce qu'une fissure longitudinale et quelles sont ses caractéristiques ?",

    "Qu'est-ce qu'une fissure transversale ?",

    "Quelles sont les principales causes des fissures de chaussée ?",

    "Quelle est la différence entre une fissure longitudinale et une fissure transversale ?",

    "Qu'est-ce que le faïençage d'une chaussée ?",

    "Qu'est-ce qu'une ornière et comment se manifeste-t-elle ?",

    "Quels sont les principaux types de dégradations des chaussées ?",

    "Comment identifier une dégradation de la chaussée à partir de son aspect visuel ?",

    "Quelles sont les classes de dommages présentes dans RDD2022 ?",

    "Quelle est la différence entre longitudinal crack et lateral crack ?"
]

for i, question in enumerate(test_questions, 1):

    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)
    print("QUESTION :", question)

    answer, sources = ask_rag(
        question,
        k=5
    )

    print("\nRÉPONSE :")
    print(answer)

    print("\nSOURCES RÉCUPÉRÉES :")

    for j, source in enumerate(sources, 1):
        print(
            f"{j}. {source['source']} "
            f"→ score : {source['score']:.4f}"
        )


TEST 1
QUESTION : Qu'est-ce qu'une fissure longitudinale et quelles sont ses caractéristiques ?

RÉPONSE :
Une fissure longitudinale est une rupture du revêtement relativement parallèle à la direction de la route, en dehors des pistes de roues, soit dans les bandes numéros 1, 3 et 5. Elle est principalement due à la fatigue de la structure de la chaussée. Les fissures longitudinales simples et intermittentes ont des ouvertures inférieures à 5 mm et leurs bords sont généralement francs et bien définis. Elles peuvent apparaître dans les bandes de roulement et sont essentiellement dues à la fatigue de la structure de la chaussée. 

Sources utilisées :
- SOURCE 1: Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt
- SOURCE 3: Cours_Pathologies_Entretien_Routes_Ouvrages_contenu_organise.txt

SOURCES RÉCUPÉRÉES :
1. Manuel_identification_degradations_chaussees_flexibles_contenu_organise.txt → score : 0.6195
2. Cours_Pathologies_Entretien_Routes_Ouvrages_contenu_orga

on passe au sauvegarde du RAG

sauvegarde le faisse

In [58]:
import os
import faiss

VECTOR_DIR = "/content/road_damage_rag/vector_db"

os.makedirs(VECTOR_DIR, exist_ok=True)

faiss.write_index(
    index,
    os.path.join(VECTOR_DIR, "index.faiss")
)

print("Index FAISS sauvegardé.")

Index FAISS sauvegardé.


sauvegarde le chunk

In [59]:
import pickle
import os

chunks_path = os.path.join(
    VECTOR_DIR,
    "chunks.pkl"
)

with open(chunks_path, "wb") as f:
    pickle.dump(chunks, f)

print("Chunks sauvegardés.")

Chunks sauvegardés.


Sauvegarder la configuration

C'est utile pour savoir comment l'index a été construit.

In [60]:
import json

config = {
    "embedding_model": "BAAI/bge-m3",
    "llm_model": "Qwen/Qwen2.5-7B-Instruct",
    "chunk_size": 800,
    "chunk_overlap": 100,
    "embedding_dimension": int(chunk_embeddings.shape[1]),
    "num_chunks": len(chunks),
    "retrieval_k": 5
}

config_path = os.path.join(
    VECTOR_DIR,
    "config.json"
)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Configuration sauvegardée.")

Configuration sauvegardée.


Vérifier les fichiers

In [61]:
import os

print("Contenu de vector_db :")

for file in os.listdir(VECTOR_DIR):
    path = os.path.join(VECTOR_DIR, file)
    print(
        file,
        "→",
        round(os.path.getsize(path) / 1024 / 1024, 2),
        "MB"
    )

Contenu de vector_db :
index.faiss → 1.87 MB
config.json → 0.0 MB
chunks.pkl → 0.39 MB


In [46]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

In [47]:
from fastapi import FastAPI
from pydantic import BaseModel


app = FastAPI(
    title="Road Damage RAG API",
    description="API RAG pour les dommages routiers",
    version="1.0"
)


class QuestionRequest(BaseModel):
    question: str
    k: int = 5


@app.get("/")
def root():

    return {
        "status": "ok",
        "message": "Road Damage RAG API is running"
    }


@app.post("/ask")
def ask(request: QuestionRequest):

    answer, sources = ask_rag(
        request.question,
        k=request.k
    )

    return {
        "answer": answer,
        "sources": [
            {
                "source": source["source"],
                "score": source["score"],
                "text": source["text"]
            }
            for source in sources
        ]
    }

In [48]:
import threading
import uvicorn


def run_api():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

print("API démarrée sur le port 8000")

API démarrée sur le port 8000


In [52]:
from pyngrok import ngrok

ngrok.set_auth_token("2yMZiUfSIUKPKEANjHF8uDISCf9_5y9NAyrMPCFkPUDyD9bWT")

In [53]:
public_url = ngrok.connect(8000)

print("URL publique :")
print(public_url)

URL publique :
NgrokTunnel: "https://5476-34-6-234-72.ngrok-free.app" -> "http://localhost:8000"
